In [ ]:
# 01 — Data Cleaning & Validation
Load the source dataset, inspect every column, fix quality issues, validate, export v3.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/clean_brick_dataset_v2.csv')
df.shape


In [ ]:
for col in df.columns:
    print(f"===== {col} =====")
    print(df[col].value_counts(dropna=False).head(15))
    print()
    

In [ ]:
dupes = df[df.duplicated(subset=['Company Name', 'Location'], keep=False)]
print(f"Rows involved in potential duplicates: {len(dupes)}")
dupes.sort_values('Company Name')[['Company Name', 'Location', 'Districts', 'Status', 'Contact']]

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=['Company Name', 'Location'], keep='first').reset_index(drop=True)
after = len(df)
print(f"Dropped {before - after} duplicate rows: {before} → {after}")

**Decision — duplicates:** 6 factory pairs appeared twice with identical name/address/phone but conflicting districts (border-area scrape overlap). Kept first occurrence; district for these 6 should be treated as approximate.

In [ ]:
# Strip whitespace from every text column (kills bug #2 everywhere, found or not)
for col in df.select_dtypes(include=['object', 'str']).columns:
    df[col] = df[col].str.strip()

# Fix casing on the three diagnosed columns
df['Company Type'] = df['Company Type'].str.title()
df['Company Category'] = df['Company Category'].str.replace('fly Ash', 'Fly Ash')
df['Status'] = df['Status'].str.title()


In [ ]:
for col in ['Company Type', 'Machine Type', 'Company Category', 'Status']:
    print(f"===== {col} =====")
    print(df[col].value_counts(dropna=False))
    print()

In [ ]:
df = df.replace('Not Specified', np.nan)

missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

In [ ]:
df['has_website'] = df['Website'].notna()
df['has_contact'] = df['Contact'].notna()

df = df.drop(columns=['Founder'])

for col in ['Machine Type', 'Product Type', 'Block Type', 'Company Category', 'Company Size']:
    df[col] = df[col].fillna('Unknown')

df.shape

In [ ]:
assert len(df) == 355, "Row count wrong"
assert df['Status'].nunique() == 3, "Status has wrong number of values"
assert not df['Districts'].isna().any(), "Districts has nulls"
assert not df['Company Type'].str.contains('manufacturing', case=True, na=False).any(), "Casing bug remains"
assert df['Machine Type'].str.startswith(' ').sum() == 0, "Whitespace bug remains"
print("All assertions passed ✓")